## Proof of Concept

PoC to generate a potential scoring system. The crux is:

1. Split by time: *Past* and *Future*
2. Define "Hipsters" as people who listened to a similar, unpopular song in the *future*
   dataset
3. Define the match score based on (2)
4. Create a list of user pairings, with the match score, where match score is above a 
   threshold

The output of 4 becomes our ground truth of users who will have good match scores

In [2]:
import pandas as pd

df = pd.read_csv("../../data/naive-sample.csv", sep=";")
df.head()

,user_id,listen_timestamp,track_mbid,track_name,artist_mbid,artist_name,album_mbid,album_name,spotify_id,duration_ms,...,liveness,loudness,speechiness,tempo,valence,mode,key,time_signature,popularity,explicit
0,13342d2b-bc47-4f13-9b68-1fa8e824f11a,2013-03-27 18:09:59 UTC,9d9fed58-4404-48b6-83b0-e087be0fa05b,Bloody Poland,de51f503-40ec-4fbe-ae22-5539b68a605a,Strachy na Lachy,b968f299-61d8-4823-908f-9dc792cbce7c,!TO!,4N8VHb6GrXy3YYPu7yhPqy,280000,...,0.5870,-6.183,0.0793,141.904,0.616,0,11,4,21,False
1,12d5ccaa-a27f-49b8-a6d2-ae9353774a37,2013-02-08 13:10:57 UTC,d2512875-0148-4b63-8f16-fc542cabce25,Girls Talkin Bout,b5ddbf00-9166-41a2-ba10-4e80b029dab6,Mindless Behavior,3b1cdcba-1192-49bd-8d13-f900e16e0c5e,#1 Girl,1l0w3qSXUh8eEkzeotScCX,202186,...,0.0882,-5.103,0.0838,122.764,0.626,0,7,4,34,False
2,12d5ccaa-a27f-49b8-a6d2-ae9353774a37,2012-08-18 23:53:23 UTC,f5fb2ecb-a9d8-482d-8a9b-1af039f37799,Missing You,b5ddbf00-9166-41a2-ba10-4e80b029dab6,Mindless Behavior,3b1cdcba-1192-49bd-8d13-f900e16e0c5e,#1 Girl,3Z0xAPKoKwE4DK2BV4chPz,222506,...,0.0448,-4.416,0.0509,120.078,0.459,1,5,4,14,False
3,12d5ccaa-a27f-49b8-a6d2-ae9353774a37,2013-01-18 11:14:15 UTC,f5fb2ecb-a9d8-482d-8a9b-1af039f37799,Missing You,b5ddbf00-9166-41a2-ba10-4e80b029dab6,Mindless Behavior,3b1cdcba-1192-49bd-8d13-f900e16e0c5e,#1 Girl,3Z0xAPKoKwE4DK2BV4chPz,222506,...,0.0448,-4.416,0.0509,120.078,0.459,1,5,4,14,False
4,12d5ccaa-a27f-49b8-a6d2-ae9353774a37,2011-12-19 18:42:18 UTC,6d851fc5-07ae-4572-9a10-ce7cbc995f06,Hook It Up,b5ddbf00-9166-41a2-ba10-4e80b029dab6,Mindless Behavior,3b1cdcba-1192-49bd-8d13-f900e16e0c5e,#1 Girl,5mOYz3EQKEkeEM5h4Qrwc9,251853,...,0.0606,-5.207,0.1020,134.894,0.616,0,9,4,17,False


In [5]:
# convert popularity to numeric (0-100). Fill missing with 50 (average)
df["popularity"] = pd.to_numeric(df["popularity"], errors="coerce").fillna(50)

# ensure timestamps are datetime (adjust `unit` or `format` as needed)
df["listen_timestamp"] = pd.to_datetime(
    df["listen_timestamp"], unit="s", errors="coerce"
)

In [6]:
# Split
# use the 80th percentile of time as our cutoff
cutoff_date = df["listen_timestamp"].quantile(0.8)

past_df = df[df["listen_timestamp"] <= cutoff_date]
future_df = df[df["listen_timestamp"] > cutoff_date]

print(f"Split Date: {cutoff_date}")
print(
    f"Past Rows (For Features): {len(past_df)} | Future Rows (For Labels): {len(future_df)}"
)


Split Date: 2012-03-26 13:30:08+00:00
Past Rows (For Features): 10876903 | Future Rows (For Labels): 2719226


In [7]:
# Edge List (Hipster Overlap)

# Get unique user-track combinations in the future to avoid over-counting a song on loop
future_tracks = future_df[["user_id", "track_mbid", "popularity"]].drop_duplicates()

# Calculate the "Hipster Weight" (Inverse Popularity)
# If popularity is 100 (Drake), weight is 0. If popularity is 5 (Obscure Indie), weight is 95.
future_tracks["hipster_weight"] = 100 - future_tracks["popularity"]

# Self-Merge on track_mbid to find users who listened to the same tracks
overlaps = pd.merge(
    future_tracks[["user_id", "track_mbid", "hipster_weight"]],
    future_tracks[["user_id", "track_mbid"]],
    on="track_mbid",
    suffixes=("_anchor", "_positive"),
)

# Remove self-matches (User 1 == User 1) AND duplicate pairs (User 1 & 2 vs User 2 & 1)
# The `<` operator cleanly halves our dataframe and solves both problems!
overlaps = overlaps[overlaps["user_id_anchor"] < overlaps["user_id_positive"]]

# Sum the hipster weights for every unique pair of users
pair_scores = (
    overlaps.groupby(["user_id_anchor", "user_id_positive"])["hipster_weight"]
    .sum()
    .reset_index()
)
pair_scores.rename(columns={"hipster_weight": "match_score"}, inplace=True)

pair_scores.head()

,user_id_anchor,user_id_positive,match_score
0,00028a63-2ba8-44fb-96d1-becb542f4a64,000375be-eeed-442d-b671-028ef6990c4d,17
1,00028a63-2ba8-44fb-96d1-becb542f4a64,0003b15e-deb6-4189-adb8-eb495f3da2ec,879
2,00028a63-2ba8-44fb-96d1-becb542f4a64,000487bd-ad15-4607-b5c3-a50c33fd3d64,521
3,00028a63-2ba8-44fb-96d1-becb542f4a64,00048e37-6f91-438a-8683-7f89c716e72f,193
4,00028a63-2ba8-44fb-96d1-becb542f4a64,0004fcab-fe30-414b-9943-243df7c0a22e,1758


In [9]:
# Filter for "High Matches"
pair_scores = pair_scores.sort_values(
    ["user_id_anchor", "match_score"], ascending=[True, False]
)

# Keep only the Top 10 "Future Friends" for every user to act as our Positive (P) labels
edge_list = pair_scores.groupby("user_id_anchor").head(7).reset_index(drop=True)

print(edge_list.shape)
edge_list.head(10)


(13550, 3)


,user_id_anchor,user_id_positive,match_score
0,00028a63-2ba8-44fb-96d1-becb542f4a64,1cd3cb84-05c1-4eae-b41a-34e7f145fc72,6075
1,00028a63-2ba8-44fb-96d1-becb542f4a64,101690a5-2f6d-43fb-8ca6-aba41395a090,5552
2,00028a63-2ba8-44fb-96d1-becb542f4a64,12c2f061-d519-4d3c-b0f7-6b38d750fdef,5424
3,00028a63-2ba8-44fb-96d1-becb542f4a64,16924dc8-137d-4366-aa07-9f958bee1e74,5334
4,00028a63-2ba8-44fb-96d1-becb542f4a64,00952afa-e8e2-4b58-8a49-8e508379afd5,4993
5,00028a63-2ba8-44fb-96d1-becb542f4a64,00d88681-bcbf-4f7f-a5db-f65509fa9257,4818
6,00028a63-2ba8-44fb-96d1-becb542f4a64,11044d09-78f7-4b31-ab2c-bc60dbb7ff74,4810
7,000375be-eeed-442d-b671-028ef6990c4d,125968db-d42d-405b-9bc9-c8ff9148d6a8,3609
8,000375be-eeed-442d-b671-028ef6990c4d,14a9b027-d3ad-41f0-8e70-5d4d2ec877f1,3571
9,000375be-eeed-442d-b671-028ef6990c4d,00ca37ad-865f-470f-a8e8-1a1f60a12737,3494


In [ ]:
edge_list.user_id_anchor.nunique() # so, all 1943 had at least one match...fair enough

1943

The idea here is that we now have a set of "good" matches from the labels. We know that:

- `00028a63-2ba8-44fb-96d1-becb542f4a64`'s best match is 
  `1cd3cb84-05c1-4eae-b41a-34e7f145fc72`
- Hipster label is _one_ example of a match. There could be others as well

In [ ]:
edge_list.to_csv(
    "../../data/edgelist.csv"   ,
    columns=["user_id_anchor", "user_id_positive", "match_score"],
    index=False,
)